### 📐 상관계수(Correlation Coefficient) 정의

두 변수 사이에 **직선 형태의 관계가 얼마나 강한지**를 -1부터 +1 사이의 숫자 하나로 나타낸 지표. 정확히는 **피어슨 상관계수(Pearson correlation coefficient)**를 의미하며, `np.corrcoef()`가 계산하는 값도 이것이다.

**수식**

$$
r = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2} \sqrt{\sum_{i=1}^{n}(y_i - \bar{y})^2}}
$$

| 기호 | 의미 |
|---|---|
| $x_i, y_i$ | 각 데이터 값 |
| $\bar{x}, \bar{y}$ | X와 Y 각각의 평균 |
| $n$ | 데이터 개수 |

**직관적 이해**
- 분자(공분산): X가 평균보다 클 때 Y도 평균보다 크면 양수가 쌓이고, X가 클 때 Y가 작으면 음수가 쌓임 → "같이 움직이는 경향"을 숫자로 표현
- 분모(표준편차의 곱): 공분산을 X, Y 각자의 변동 크기로 나눠서 단위와 무관하게 **-1~1로 정규화**

**값의 해석**

| 범위 | 해석 |
|---|---|
| r = 1 | 완벽한 양의 선형관계 |
| r = -1 | 완벽한 음의 선형관계 |
| r = 0 | 선형관계 없음 |
| 0.7 ~ 1.0 | 강한 양의 상관 |
| 0.3 ~ 0.7 | 중간 정도의 양의 상관 |
| 0 ~ 0.3 | 약한 상관 |

(음수 쪽도 대칭으로 동일하게 해석)

> ⚠️ **주의**: 상관계수는 **선형(직선) 관계만** 포착한다. U자 곡선처럼 비선형 관계는 실제로 강한 관계여도 상관계수가 0에 가깝게 나올 수 있다. 상관계수를 보기 전에 **산점도(scatter plot)를 먼저 그려보는 습관**이 중요하다.

### 모집단과 표본집단

**모집단(Population)**
연구/분석의 대상이 되는 **전체 집단**. 알고 싶은 대상 전부를 의미한다.
> 예: 대한민국 전체 성인의 평균 키

**표본집단(Sample)**
모집단 전체를 조사하기 어려울 때, 그 중 **일부를 뽑아** 조사한 집단.
> 예: 대한민국 성인 1,000명을 뽑아 조사한 키 데이터

**둘의 관계**
모집단을 전부 조사하는 건 시간과 비용이 많이 들기 때문에, 표본을 뽑아 조사한 뒤 그 결과로 **모집단 전체를 추정**한다.

| 구분 | 대상 | 예시 |
|---|---|---|
| 모집단 | 전체 | 전교생 1,000명 |
| 표본집단 | 일부 | 무작위로 뽑은 100명 |

> 표본은 모집단을 대표할 수 있도록 **무작위(랜덤)로 뽑는 것**이 중요하다. 특정 집단에 치우쳐 뽑으면(예: 농구부 학생만 뽑아서 키 조사) 표본이 모집단을 제대로 대표하지 못한다.

In [5]:
import numpy as np
import oracledb
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

USER = os.getenv("USER")
PASSWORD = os.getenv("PASSWORD")
DSN = os.getenv("DSN")

conn = oracledb.connect(user=USER, password=PASSWORD, dsn=DSN)
query = "SELECT * FROM DELIVERY_ORDERS"
df = pd.read_sql(query, conn)

print("=== 지금 우리가 가진 데이터는 '표본'이라고 가정 ===")
print(f"표본 크기(n) = {len(df)}건")

=== 지금 우리가 가진 데이터는 '표본'이라고 가정 ===
표본 크기(n) = 37건


C:\Users\lkw\AppData\Local\Temp\ipykernel_10756\3126001115.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


## 표준편차 계산 시 나누는 방법이 달라지는 이유

표준편차 = "데이터가 평균에서 얼마나 흩어져 있는지"를 나타내는 숫자

| 구분 | 나누는 값 | 옵션 |
|---|---|---|
| 모집단 표준편차 | 데이터 개수(n) | `ddof=0` |
| 표본 표준편차 | 데이터 개수보다 1개 적게(n-1) | `ddof=1` |

In [6]:
price = df["PRICE"].values

# ddof = 0: 이 데이터가 전체(모집단)이라고 가정
# std : 표준편차
# ddof = 1 : 이 데이터가 일부(표본)이라고 가정
std_population = np.std(price, ddof=0)
std_sample = np.std(price, ddof=1)

print("=== 가격(PRICE)의 표준편차 비교")
print(f"모집단으로 가정(ddof = 0) : {std_population:.2f}원")
print(f"표본으로 가정(ddof = 1) : {std_sample:.2f}원")
print(f"차이 : {std_sample - std_population:.2f}원")

=== 가격(PRICE)의 표준편차 비교
모집단으로 가정(ddof = 0) : 7663.10원
표본으로 가정(ddof = 1) : 7768.81원
차이 : 105.70원


In [ ]:
# numpy np.std()의 기본값은 ddof=0 (모집단으로 가정)
# pandas .std()의 기본값은 ddof=1 (표본으로 가정)


print("=== pandas와 numpy 기본값 비교 ===")
print("pandas df['PRICE'].std()  :", df["PRICE"].std())          # pandas는 기본적으로 ddof = 1
print("numpy np.std(price)  :", np.std(price))     # 기본 ddof = 0
print("numpy np.std(price, ddof = 1) :", np.std(price, ddof=1))    # pandas의 분석과 동일해짐

print(f"\n 현재 데이터의 개수(n)= {len(price)}건")
print("데이터가 적을수록 두 계산 방식의 차이가 크게 벌어짐. 즉 모집단과 표준집단의 차이가 커서 불안정성이 높아짐")

=== pandas와 numpy 기본값 비교 ===
pandas df['PRICE'].std()  : 7768.806494845547
numpy np.std(price)  : 7663.103521943342
numpy np.std(price, ddof = 1) : 7768.806494845547

 현재 데이터의 개수(n)= 37건


### 기초 통계 요약 (평균, 최댓값, 최솟값)

In [16]:
rating = df["RATING"].values

print("=== 가격 통계 요약 ===")
print(f"평균 : {np.mean(price):.0f}원")
print(f"최댓값 : {np.max(price)}원")
print(f"최솟값 : {np.min(price)}원")

print("=== 평점 통계 요약 ===")
print(f"평균 : {np.mean(rating):.2f}원")
print(f"최댓값 : {np.max(rating):.2f}원")
print(f"최솟값 : {np.min(rating):.2f}원")


=== 가격 통계 요약 ===
평균 : 15581원
최댓값 : 29000원
최솟값 : 5000원
=== 평점 통계 요약 ===
평균 : 4.38원
최댓값 : 4.90원
최솟값 : 3.90원


In [18]:
corr_matrix = np.corrcoef(price, rating)
print("=== 상관계수 행렬(2x2) ===")
print(corr_matrix)                                  

# 행렬에서 대각선(1.0)은 "자기 자신과의 상관관계"라서 항상 1
# 나머지가 진짜 (0.3 ~ 0.7이면 중간정도의 상관관계 -> 같이 가다가 한 두번 다른 값)

corr_value = corr_matrix[0,1]
print(f"\n가격-평점 상관계수 : {corr_value:.3f}")

=== 상관계수 행렬(2x2) ===
[[1.         0.49015948]
 [0.49015948 1.        ]]

가격-평점 상관계수 : 0.490


In [19]:
# 상관계수 숫자를 '친구 관계'로 해석

if corr_value >= 0.7:
    friendship = "죽마고우(거의 항상 같이 움직임)"
elif corr_value >= 0.3:
    friendship = "친한 친구(대체로 같이 움직지만 가끔 따로 놈)"
elif corr_value >= -0.3:
    friendship = "그냥 반 친구(같이 다닐 때도 있고 아닐때도 랜덤)"
else :
    friendship = "앙숙 (반대로 다)"

print(f"비유로 해석하면 -> {friendship}")

비유로 해석하면 -> 친한 친구(대체로 같이 움직지만 가끔 따로 놈)


In [ ]:
# 카테고리별로 나눠서 상관계수 다시 보기

print("=== 카테고리별 가격-평점 상관계수 ===")
for category in df["CATEGORY"].unique():
    sub_df = df[df["CATEGORY"] == category]
    print(sub_df)   
    if len(sub_df) >= 3:
        c = np.corrcoef(sub_df["PRICE"], sub_df["RATING"])[0,1]      # 2x2 행렬중 0행 1열 값
        print(f"{category:6s} (n={len(sub_df):2d}건) : {c:6.3f}")
    else:
        print(f"{category:6s} : 데이터가 너무 적어서 상관계수 계산을 생략합니다 (n={len(sub_df)})")

=== 카테고리별 가격-평점 상관계수 ===
분식     (n= 7건) :  0.891
치킨     (n= 6건) :  0.502
피자     (n= 6건) :  0.851
중식     (n=10건) :  0.764
일식     (n= 8건) :  0.961
